In [ ]:
import json
import pandas as pd

#import json — Python has a built-in tool called json that knows how to read .json files. We're loading it here so we can use it later.
#import pandas as pd — Pandas is the library that creates tables (called DataFrames) in Python. as pd just means instead of typing pandas every time, we type pd. Just a shortcut.

In [1]:
def parse_match(filepath):
    with open(filepath) as f:
        match = json.load(f)

    info = match['info']
    teams = info['teams']
    winner = info.get('outcome', {}).get('winner', None)
    match_id = filepath.split('/')[-1].replace('.json', '')
    date = info['dates'][0]
    venue = info.get('venue', '')
    city = info.get('city', '')
    toss_winner = info['toss']['winner']
    toss_decision = info['toss']['decision']

    rows = []

    for inning_idx, inning in enumerate(match['innings']):
        batting_team = inning['team']
        bowling_team = [t for t in teams if t != batting_team][0]

        # Target is stored inside 2nd innings object
        target = None
        if 'target' in inning:
            target = inning['target']['runs']

        legal_ball_count = 0
        runs_so_far = 0
        wickets_so_far = 0

        for over_data in inning.get('overs', []):
            over_num = over_data['over']

            for delivery in over_data['deliveries']:
                extras = delivery.get('extras', {})

                # A legal ball = not a wide and not a no-ball
                is_wide = 'wides' in extras
                is_noball = 'noballs' in extras
                is_legal = not is_wide and not is_noball

                if is_legal:
                    legal_ball_count += 1

                runs_this_ball = delivery['runs']['total']
                runs_so_far += runs_this_ball

                wickets_this_ball = delivery.get('wickets', [])
                is_wicket = len(wickets_this_ball) > 0
                if is_wicket:
                    wickets_so_far += 1

                rows.append({
                    'match_id':         match_id,
                    'date':             date,
                    'venue':            venue,
                    'city':             city,
                    'inning':           inning_idx + 1,
                    'batting_team':     batting_team,
                    'bowling_team':     bowling_team,
                    'over':             over_num,
                    'legal_ball':       legal_ball_count,
                    'is_legal_ball':    int(is_legal),
                    'runs_this_ball':   runs_this_ball,
                    'batter_runs':      delivery['runs']['batter'],
                    'is_wide':          int(is_wide),
                    'is_noball':        int(is_noball),
                    'is_wicket':        int(is_wicket),
                    'wicket_kind':      wickets_this_ball[0]['kind'] if is_wicket else None,
                    'runs_so_far':      runs_so_far,
                    'wickets_so_far':   wickets_so_far,
                    'target':           target,
                    'toss_winner':      toss_winner,
                    'toss_decision':    toss_decision,
                    'winner':           winner,
                })

    return pd.DataFrame(rows)



In [4]:

# Test on your file — update path to match where you saved it
df = parse_match('../data/raw/335982.json')
print(df.shape)
print(df[['inning','over','legal_ball','runs_so_far','wickets_so_far','is_wicket','target']].head(30))

(225, 22)
    inning  over  legal_ball  runs_so_far  wickets_so_far  is_wicket  target
0        1     0           1            1               0          0     NaN
1        1     0           2            1               0          0     NaN
2        1     0           2            2               0          0     NaN
3        1     0           3            2               0          0     NaN
4        1     0           4            2               0          0     NaN
5        1     0           5            2               0          0     NaN
6        1     0           6            3               0          0     NaN
7        1     1           7            3               0          0     NaN
8        1     1           8            7               0          0     NaN
9        1     1           9           11               0          0     NaN
10       1     1          10           17               0          0     NaN
11       1     1          11           21               0         

In [5]:
# Check innings 2
inn2 = df[df['inning'] == 2]
print("Innings 2 rows:", len(inn2))
print("Target value:", inn2['target'].iloc[0])
print("Final score inn1:", df[df['inning']==1]['runs_so_far'].max())
print("Final score inn2:", inn2['runs_so_far'].max())
print("Winner:", df['winner'].iloc[0])
print("\nWickets that fell:")
print(df[df['is_wicket']==1][['inning','over','legal_ball','runs_so_far','wickets_so_far']])

Innings 2 rows: 101
Target value: 223.0
Final score inn1: 222
Final score inn2: 82
Winner: Kolkata Knight Riders

Wickets that fell:
     inning  over  legal_ball  runs_so_far  wickets_so_far
33        1     5          32           61               1
74        1    12          73          112               2
106       1    17         103          172               3
131       2     1           7            4               1
138       2     2          14            9               2
154       2     4          29           24               3
157       2     5          32           24               4
174       2     7          47           38               5
177       2     8          50           38               6
183       2     8          54           43               7
197       2    11          67           57               8
210       2    13          79           70               9
224       2    15          91           82              10


In [7]:
import os
from tqdm import tqdm

def parse_all_matches(raw_dir):
    all_rows = []
    files = [f for f in os.listdir(raw_dir) if f.endswith('.json')]
    print(f"Total files found: {len(files)}")
    
    for f in tqdm(files):
        try:
            df_match = parse_match(os.path.join(raw_dir, f))
            all_rows.append(df_match)
        except Exception as e:
            print(f"Skipped {f} — {e}")
    
    final_df = pd.concat(all_rows, ignore_index=True)
    return final_df

# Run it
df_all = parse_all_matches('../data/raw')
print(f"\nTotal rows: {df_all.shape[0]}")
print(f"Total columns: {df_all.shape[1]}")
print(f"Matches parsed: {df_all['match_id'].nunique()}")

Total files found: 1241


100%|██████████| 1241/1241 [00:25<00:00, 48.83it/s]



Total rows: 295258
Total columns: 22
Matches parsed: 1241


In [8]:
df_all.to_csv('../data/processed/all_matches.csv', index=False)
print("Saved!")

Saved!


In [11]:
print(df_all.head())
print("\nNull values per column:")
print(df_all.isnull().sum())
print("\nMatches per season:")
print(df_all.groupby(df_all['date'].str[:4])['match_id'].nunique().sort_index())

  match_id        date                                      venue       city  \
0  1082591  2017-04-05  Rajiv Gandhi International Stadium, Uppal  Hyderabad   
1  1082591  2017-04-05  Rajiv Gandhi International Stadium, Uppal  Hyderabad   
2  1082591  2017-04-05  Rajiv Gandhi International Stadium, Uppal  Hyderabad   
3  1082591  2017-04-05  Rajiv Gandhi International Stadium, Uppal  Hyderabad   
4  1082591  2017-04-05  Rajiv Gandhi International Stadium, Uppal  Hyderabad   

   inning         batting_team                 bowling_team  over  legal_ball  \
0       1  Sunrisers Hyderabad  Royal Challengers Bangalore     0           1   
1       1  Sunrisers Hyderabad  Royal Challengers Bangalore     0           2   
2       1  Sunrisers Hyderabad  Royal Challengers Bangalore     0           3   
3       1  Sunrisers Hyderabad  Royal Challengers Bangalore     0           4   
4       1  Sunrisers Hyderabad  Royal Challengers Bangalore     0           4   

   is_legal_ball  ...  is_wide  

In [10]:
df_all['match_id'] = df_all['match_id'].str.replace('raw\\', '', regex=False)
df_all.to_csv('../data/processed/all_matches.csv', index=False)
print("match_id cleaned and saved!")
print(df_all['match_id'].head())

match_id cleaned and saved!
0    1082591
1    1082591
2    1082591
3    1082591
4    1082591
Name: match_id, dtype: str


In [12]:
print(df_all.isnull().sum())


match_id               0
date                   0
venue                  0
city                   0
inning                 0
batting_team           0
bowling_team           0
over                   0
legal_ball             0
is_legal_ball          0
runs_this_ball         0
batter_runs            0
is_wide                0
is_noball              0
is_wicket              0
wicket_kind       280575
runs_so_far            0
wickets_so_far         0
target            153183
toss_winner            0
toss_decision          0
winner              4980
dtype: int64
